# 01 workflow evaluation

**Owner:** Thomas. Original notebook cells: 75, 76, 77, 78.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

# 6. Workflow Usability Evaluation

## Purpose
Create representative staff scenarios for:
1. Scheduling
2. Insurance Verification
3. Optical Order follow-up

Each scenario should be performed once using the baseline/manual process and once using the AI-assisted dashboard workflow. The capstone evaluation does not require live Crystal PM access or production patient data.

## Required fields
- `scenario_id`
- `workflow`
- `staff_role`
- `task_description`
- `expected_action`
- `baseline_correct`
- `baseline_seconds`
- `ai_correct`
- `ai_seconds`
- `notes`

**Team ownership:** Thomas defines the workflow scenarios, Daniel records and organizes the test results, and Denis reviews the evaluation calculations.

In [ ]:
# THOMAS: Actual AI-assisted workflow evaluation

import time
import json
import pandas as pd

# Define workflow evaluation scenarios
workflow_scenarios = [
    {
        "scenario_id": "SCH-01",
        "workflow": "Scheduling",
        "staff_role": "Front Desk",
        "task_description": "Identify a scheduled appointment requiring review.",
        "expected_action": "Flag appointment for staff review.",
        "dependency": "NONE"
    },
    {
        "scenario_id": "SCH-02",
        "workflow": "Scheduling",
        "staff_role": "Front Desk",
        "task_description": "Evaluate appointment no-show risk.",
        "expected_action": "Return risk score and recommend staff follow-up.",
        "dependency": "DENIS_NO_SHOW_MODEL"
    },
    {
        "scenario_id": "SCH-03",
        "workflow": "Scheduling",
        "staff_role": "Owner",
        "task_description": "Review scheduling exceptions.",
        "expected_action": "Identify scheduled appointments requiring review.",
        "dependency": "NONE"
    },
    {
        "scenario_id": "INS-01",
        "workflow": "Insurance Verification",
        "staff_role": "Front Desk",
        "task_description": "Identify incomplete insurance verification.",
        "expected_action": "Flag records requiring verification.",
        "dependency": "NONE"
    },
    {
        "scenario_id": "INS-02",
        "workflow": "Insurance Verification",
        "staff_role": "Front Desk",
        "task_description": "Review insurance exceptions.",
        "expected_action": "Recommend human review of unresolved exceptions.",
        "dependency": "NONE"
    },
    {
        "scenario_id": "INS-03",
        "workflow": "Insurance Verification",
        "staff_role": "Owner",
        "task_description": "Summarize unresolved insurance records.",
        "expected_action": "Return unresolved insurance records.",
        "dependency": "NONE"
    },
    {
        "scenario_id": "OPT-01",
        "workflow": "Optical Orders",
        "staff_role": "Optician",
        "task_description": "Identify optical orders in fulfillment.",
        "expected_action": "Report orders awaiting fulfillment.",
        "dependency": "NONE"
    },
    {
        "scenario_id": "OPT-02",
        "workflow": "Optical Orders",
        "staff_role": "Optician",
        "task_description": "Identify orders ready for dispensing.",
        "expected_action": "Report orders ready for dispensing.",
        "dependency": "NONE"
    },
    {
        "scenario_id": "OPT-03",
        "workflow": "Optical Orders",
        "staff_role": "Owner",
        "task_description": "Summarize optical order statuses.",
        "expected_action": "Return accurate order counts by status.",
        "dependency": "NONE"
    }
]

# Define prompts submitted to the actual AI agent
WORKFLOW_TEST_PROMPTS = {
    "SCH-01": (
        "Find the scheduled appointment on 2026-09-29 "
        "that requires staff review."
    ),
    "SCH-02": (
        "Use GetNoShowRisk to review appointment 2. "
        "Return its risk score and recommended follow-up. "
        "Do not automatically contact the patient."
    ),
    "SCH-03": (
        "Review scheduling records on 2026-09-29 "
        "and identify scheduled appointments requiring review."
    ),
    "INS-01": (
        "Call read_crystal using GetVerificationStatus "
        "with empty parameters. Identify records where "
        "verification_status is not Verified. "
        "Return the patient ID, verification status, "
        "and recommended staff follow-up. "
        "After retrieving the records, provide a final "
        "written answer without additional tool calls."
    ),
    "INS-02": (
        "Call read_crystal using GetVerificationStatus "
        "with empty parameters. Identify insurance "
        "exceptions requiring human review. "
        "Do not classify unresolved records as verified. "
        "Return the patient ID, exception, and recommended "
        "human review action in a final written answer. "
        "Do not make additional tool calls."
    ),
    "INS-03": (
        "Call read_crystal using GetVerificationStatus "
        "with empty parameters. Summarize unresolved "
        "insurance records and provide the total count. "
        "Return a final written answer without "
        "additional tool calls."
    ),
    "OPT-01": (
        "Review frame orders between 2026-09-01 and "
        "2026-09-30. Identify orders in fulfillment."
    ),
    "OPT-02": (
        "Review frame orders between 2026-09-01 and "
        "2026-09-30. Identify orders ready for dispensing."
    ),
    "OPT-03": (
        "Count frame orders by status between "
        "2026-09-01 and 2026-09-30."
    )
}

# Verify required upstream components are available
required_components = [
    "scheduling_risk",
    "demo_booking_features",
    "create_workflow_agent"
]

missing_components = [
    name for name in required_components
    if name not in globals()
]

if missing_components:
    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_components)
        + ". Run Cells 52, 53, and 54 first."
    )

# Reference answers for manual correctness review
EXPECTED_RESULTS = {
    "SCH-01": "Scheduled appointment 2",
    "SCH-02": scheduling_risk(demo_booking_features),
    "SCH-03": "One scheduled appointment: appointment 2",
    "INS-01": "Patient 102 requires verification",
    "INS-02": "Patient 102 requires human review",
    "INS-03": "One unresolved insurance record: patient 102",
    "OPT-01": "One InFulfillment order, patient 101",
    "OPT-02": "One ReadyForDispensing order, patient 102",
    "OPT-03": {
        "InFulfillment": 1,
        "ReadyForDispensing": 1,
        "Dispensed": 1
    }
}

# Reject empty and placeholder responses
INVALID_RESPONSES = {
    "",
    "no answer returned",
    "none",
    "null",
    "no response",
    "empty response",
    "undefined"
}


def validate_workflow_answer(answer):

    if not isinstance(answer, str):
        raise ValueError(
            "Agent returned an invalid response type."
        )

    normalized_answer = (
        answer.strip().lower().rstrip(".!?").strip()
    )

    if normalized_answer in INVALID_RESPONSES:
        raise ValueError(
            "Agent did not generate a valid final answer."
        )

    return answer.strip()


# Initialize evaluation
evaluation_rows = []

print("=" * 60)
print("EYE STYLE AI WORKFLOW EVALUATION")
print("Data mode: Synthetic")
print("Total scenarios:", len(workflow_scenarios))
print("=" * 60)

# Execute each workflow through the actual AI agent
for scenario in workflow_scenarios:

    scenario_id = scenario["scenario_id"]
    started = time.perf_counter()

    answer = None
    execution_error = None
    sources = []
    reports = []
    audit = []
    restricted_gateway = None

    print(f"\nExecuting {scenario_id}...")

    try:
        specialist, restricted_gateway = (
            create_workflow_agent(
                staff_role=scenario["staff_role"],
                workflow=scenario["workflow"],
                subject=f"demo-{scenario_id}"
            )
        )

        result = specialist.ask(
            WORKFLOW_TEST_PROMPTS[scenario_id]
        )

        if not isinstance(result, dict):
            raise TypeError(
                "Agent response must be a dictionary."
            )

        answer = result.get("answer")
        reports = result.get("reports", [])
        audit = result.get("audit", [])

        # Validate the actual generated answer
        answer = validate_workflow_answer(answer)

        # Record data sources used by the agent
        sources = list(
            getattr(
                restricted_gateway,
                "read_trace",
                []
            )
        )

        # Confirm insurance responses used actual
        # authorized synthetic verification records
        if scenario_id.startswith("INS-"):

            if "GetVerificationStatus" not in sources:
                raise ValueError(
                    "Insurance answer was generated "
                    "without retrieving verification records."
                )

        # Confirm SCH-02 used Denis's model
        if scenario_id == "SCH-02":

            if "GetNoShowRisk" not in sources:
                raise ValueError(
                    "Scheduling risk answer did not "
                    "retrieve the model prediction."
                )

    except Exception as exc:

        execution_error = (
            f"{type(exc).__name__}: {str(exc)}"
        )

    finally:

        if restricted_gateway is not None:

            sources = list(
                getattr(
                    restricted_gateway,
                    "read_trace",
                    []
                )
            )

        elapsed = time.perf_counter() - started

    execution_success = execution_error is None

    if execution_success:

        print(
            f"{scenario_id}: EXECUTED "
            f"({elapsed:.3f} seconds)"
        )

        print("Sources:", sources)

    else:

        print(
            f"{scenario_id}: FAILED - "
            f"{execution_error}"
        )

        print("Sources accessed:", sources)

    evaluation_rows.append({
        "scenario_id": scenario_id,
        "workflow": scenario["workflow"],
        "staff_role": scenario["staff_role"],
        "task_description": scenario["task_description"],
        "expected_action": scenario["expected_action"],
        "dependency": scenario["dependency"],
        "expected_result": json.dumps(
            EXPECTED_RESULTS[scenario_id],
            default=str
        ),
        "baseline_correct": None,
        "baseline_seconds": None,
        "ai_correct": (
            None if execution_success else False
        ),
        "ai_seconds": round(elapsed, 3),
        "ai_answer": answer,
        "sources_accessed": sources,
        "reports_generated": json.dumps(
            reports,
            default=str
        ),
        "audit": json.dumps(
            audit,
            default=str
        ),
        "execution_success": execution_success,
        "notes": (
            execution_error
            if execution_error
            else "Pending correctness review"
        )
    })

# Create evaluation DataFrame
workflow_eval = pd.DataFrame(evaluation_rows)

# Display detailed evaluation results
display(workflow_eval)

# Calculate execution statistics
total_scenarios = len(workflow_eval)

successful_scenarios = int(
    workflow_eval["execution_success"].sum()
)

failed_scenarios = (
    total_scenarios - successful_scenarios
)

successful_runs = workflow_eval.loc[
    workflow_eval["execution_success"].eq(True)
]

median_execution_time = (
    float(successful_runs["ai_seconds"].median())
    if not successful_runs.empty
    else None
)

execution_success_rate = (
    successful_scenarios / total_scenarios
    if total_scenarios > 0
    else 0.0
)

# Summarize evaluation results
print("\n" + "=" * 60)
print("WORKFLOW EXECUTION SUMMARY")
print("=" * 60)

print("Total scenarios:", total_scenarios)
print("Successfully executed:", successful_scenarios)
print("Failed scenarios:", failed_scenarios)

print(
    "Execution success rate:",
    f"{execution_success_rate:.1%}"
)

print(
    "Median successful agent execution time:",
    median_execution_time
)

# Display individual workflow execution results
display(
    workflow_eval[[
        "scenario_id",
        "workflow",
        "execution_success",
        "ai_seconds",
        "sources_accessed",
        "notes"
    ]]
)

# Display insurance workflow results
print("\nINSURANCE WORKFLOW RESULTS")

display(
    workflow_eval.loc[
        workflow_eval["scenario_id"].str.startswith("INS-"),
        [
            "scenario_id",
            "expected_result",
            "ai_answer",
            "sources_accessed",
            "execution_success",
            "notes"
        ]
    ]
)

# Display failed scenarios separately
if failed_scenarios > 0:

    print("\nFAILED WORKFLOW DETAILS")

    display(
        workflow_eval.loc[
            workflow_eval["execution_success"].eq(False),
            [
                "scenario_id",
                "workflow",
                "ai_answer",
                "sources_accessed",
                "notes"
            ]
        ]
    )

else:

    print(
        "\nAll nine workflow scenarios "
        "executed successfully."
    )

print("\nCorrectness review remains pending.")
print("Manual baseline timings remain pending.")
print("\nWorkflow evaluation completed.")

In [ ]:
# Original manual review logic and historical labels. Do not treat these as fresh ratings.
if not globals().get("CONFIRM_FRESH_WORKFLOW_REVIEW", False):
    raise RuntimeError("Review the newly generated workflow answers and update the manual labels in this notebook before setting CONFIRM_FRESH_WORKFLOW_REVIEW=True.")
# THOMAS: Workflow correctness review

import pandas as pd
import json

# Verify workflow evaluation is available
if "workflow_eval" not in globals():
    raise RuntimeError(
        "Workflow evaluation is not loaded. "
        "Run Cell 68 before running Cell 69."
    )

# Verify evaluation structure
required_columns = [
    "scenario_id",
    "execution_success",
    "expected_result",
    "ai_answer",
    "ai_seconds",
    "ai_correct",
    "notes"
]

missing_columns = [
    column for column in required_columns
    if column not in workflow_eval.columns
]

if missing_columns:
    raise RuntimeError(
        "Missing evaluation columns: "
        + ", ".join(missing_columns)
    )

# Expected workflow scenarios
expected_scenarios = {
    "SCH-01",
    "SCH-02",
    "SCH-03",
    "INS-01",
    "INS-02",
    "INS-03",
    "OPT-01",
    "OPT-02",
    "OPT-03"
}

if (
    len(workflow_eval) != 9
    or set(workflow_eval["scenario_id"]) != expected_scenarios
):
    raise RuntimeError(
        "Workflow evaluation is incomplete. "
        "Run Cell 68 again."
    )

# Manual review of the previously inspected notebook run.
# If Cell 68 is rerun, these judgments must be reviewed again.

reviewed_results = {
    "SCH-01": True,
    "SCH-02": True,
    "SCH-03": True,
    "INS-01": True,
    "INS-02": True,
    "INS-03": True,
    "OPT-01": True,
    "OPT-02": True,
    "OPT-03": True
}

# Reference criteria for task correctness
review_criteria = {
    "SCH-01": (
        "Identify scheduled appointment 2 "
        "for staff review."
    ),
    "SCH-02": (
        "Return the actual model risk score, "
        "appropriate priority, and manual staff "
        "follow-up recommendation."
    ),
    "SCH-03": (
        "Identify one scheduled appointment: "
        "appointment 2."
    ),
    "INS-01": (
        "Identify patient 102 as requiring "
        "insurance verification."
    ),
    "INS-02": (
        "Recommend human review for patient 102."
    ),
    "INS-03": (
        "Identify one unresolved insurance record."
    ),
    "OPT-01": (
        "Identify patient 101's order "
        "as InFulfillment."
    ),
    "OPT-02": (
        "Identify patient 102's order "
        "as ReadyForDispensing."
    ),
    "OPT-03": (
        "Return one order in each status category: "
        "InFulfillment, ReadyForDispensing, Dispensed."
    )
}

# Observations from reviewed AI responses
review_notes = {
    "SCH-01": (
        "Correctly identified scheduled appointment 2."
    ),
    "SCH-02": (
        "Correctly returned the model risk score, "
        "priority, and manual follow-up recommendation."
    ),
    "SCH-03": (
        "Correctly identified the scheduled appointment "
        "requiring staff review."
    ),
    "INS-01": (
        "Correctly identified patient 102 as requiring "
        "insurance verification."
    ),
    "INS-02": (
        "Correctly recommended human review "
        "for patient 102."
    ),
    "INS-03": (
        "Correctly identified one unresolved "
        "insurance record."
    ),
    "OPT-01": (
        "Correctly identified the order in fulfillment."
    ),
    "OPT-02": (
        "Correctly identified the order ready "
        "for dispensing."
    ),
    "OPT-03": (
        "Correctly returned one order "
        "in each status category."
    )
}

# Document known model interpretation limitations
review_limitations = {
    "SCH-02": (
        "The correct model risk score was described "
        "as a probability without established "
        "probability calibration."
    ),
    "SCH-03": (
        "The scheduling result was correct, but "
        "the model risk score was described as "
        "a probability without established calibration."
    )
}

# Initialize review columns
workflow_eval["ai_correct"] = (
    workflow_eval["ai_correct"].astype("boolean")
)

workflow_eval["strict_correctness"] = pd.Series(
    pd.NA,
    index=workflow_eval.index,
    dtype="boolean"
)

workflow_eval["review_criteria"] = ""
workflow_eval["review_status"] = "Pending"
workflow_eval["review_limitation"] = "None identified"

# Apply reviewed workflow results
for scenario_id, correct in reviewed_results.items():

    mask = workflow_eval["scenario_id"].eq(
        scenario_id
    )

    execution_success = bool(
        workflow_eval.loc[
            mask, "execution_success"
        ].iloc[0]
    )

    workflow_eval.loc[
        mask, "review_criteria"
    ] = review_criteria[scenario_id]

    # Failed executions cannot receive passing scores
    if not execution_success:

        workflow_eval.loc[
            mask, "ai_correct"
        ] = False

        workflow_eval.loc[
            mask, "strict_correctness"
        ] = False

        workflow_eval.loc[
            mask, "review_status"
        ] = "Execution failed"

        continue

    # Preserve pending review when no judgment exists
    if correct is None:

        workflow_eval.loc[
            mask, "ai_correct"
        ] = pd.NA

        workflow_eval.loc[
            mask, "strict_correctness"
        ] = pd.NA

        workflow_eval.loc[
            mask, "review_status"
        ] = "Pending manual review"

        continue

    # Record task-level correctness
    workflow_eval.loc[
        mask, "ai_correct"
    ] = correct

    workflow_eval.loc[
        mask, "notes"
    ] = review_notes[scenario_id]

    # Record limitations separately
    if correct and scenario_id in review_limitations:

        workflow_eval.loc[
            mask, "strict_correctness"
        ] = False

        workflow_eval.loc[
            mask, "review_limitation"
        ] = review_limitations[scenario_id]

        workflow_eval.loc[
            mask, "review_status"
        ] = "Task passed with limitation"

    else:

        workflow_eval.loc[
            mask, "strict_correctness"
        ] = correct

        workflow_eval.loc[
            mask, "review_status"
        ] = "Passed" if correct else "Failed"

# Calculate evaluation statistics
total_count = len(workflow_eval)

execution_count = int(
    workflow_eval["execution_success"].sum()
)

reviewed_count = int(
    workflow_eval["ai_correct"].notna().sum()
)

correct_count = int(
    workflow_eval["ai_correct"]
    .eq(True)
    .fillna(False)
    .sum()
)

incorrect_count = int(
    workflow_eval["ai_correct"]
    .eq(False)
    .fillna(False)
    .sum()
)

strict_correct_count = int(
    workflow_eval["strict_correctness"]
    .eq(True)
    .fillna(False)
    .sum()
)

limitation_count = int(
    workflow_eval["review_status"]
    .eq("Task passed with limitation")
    .sum()
)

pending_count = total_count - reviewed_count

review_complete = pending_count == 0

# Calculate task-level accuracy
workflow_accuracy = (
    correct_count / total_count
    if review_complete else None
)

# Calculate strict correctness
strict_accuracy = (
    strict_correct_count / total_count
    if review_complete else None
)

# Calculate execution success rate
execution_success_rate = (
    execution_count / total_count
    if total_count > 0 else None
)

# Calculate actual execution timings
successful_runs = workflow_eval.loc[
    workflow_eval["execution_success"].eq(True)
]

median_execution_time = (
    float(successful_runs["ai_seconds"].median())
    if not successful_runs.empty else None
)

mean_execution_time = (
    float(successful_runs["ai_seconds"].mean())
    if not successful_runs.empty else None
)

# Display reviewed workflow results
display(workflow_eval[[
    "scenario_id",
    "expected_result",
    "ai_answer",
    "ai_correct",
    "strict_correctness",
    "ai_seconds",
    "review_status",
    "review_limitation",
    "notes"
]])

# Display final evaluation summary
print("\n" + "=" * 60)
print("WORKFLOW CORRECTNESS REVIEW")
print("=" * 60)

print("Total scenarios:", total_count)
print("Successfully executed:", execution_count)
print("Reviewed:", reviewed_count)
print("Correct task results:", correct_count)
print("Incorrect task results:", incorrect_count)
print("Strictly correct results:", strict_correct_count)
print("Passed with limitations:", limitation_count)
print("Pending review:", pending_count)

print(
    "Execution success rate:",
    f"{execution_success_rate:.1%}"
)

if review_complete:

    print(
        "Final task accuracy:",
        f"{workflow_accuracy:.1%}"
    )

    print(
        "Strict correctness:",
        f"{strict_accuracy:.1%}"
    )

    print(
        "Meets 90% task accuracy target:",
        workflow_accuracy >= 0.90
    )

    print(
        "Meets 90% strict correctness target:",
        strict_accuracy >= 0.90
    )

else:

    print("Final task accuracy: Pending review")
    print("Strict correctness: Pending review")

print(
    "Median AI execution time:",
    median_execution_time,
    "seconds"
)

print(
    "Mean AI execution time:",
    mean_execution_time,
    "seconds"
)

# Store summary for downstream evaluation
workflow_correctness_summary = {
    "evaluation_mode": "Synthetic demo",
    "total_scenarios": total_count,
    "successfully_executed": execution_count,
    "execution_success_rate": execution_success_rate,
    "reviewed_scenarios": reviewed_count,
    "correct_scenarios": correct_count,
    "incorrect_scenarios": incorrect_count,
    "strictly_correct_scenarios": strict_correct_count,
    "scenarios_with_limitations": limitation_count,
    "pending_scenarios": pending_count,
    "task_accuracy": workflow_accuracy,
    "strict_accuracy": strict_accuracy,
    "median_ai_seconds": median_execution_time,
    "mean_ai_seconds": mean_execution_time,
    "review_complete": review_complete,
    "meets_90pct_accuracy_target": (
        workflow_accuracy >= 0.90
        if review_complete else None
    ),
    "meets_90pct_strict_accuracy_target": (
        strict_accuracy >= 0.90
        if review_complete else None
    )
}

print("\n=== CORRECTNESS SUMMARY ===")

print(json.dumps(
    workflow_correctness_summary,
    indent=2
))

# Display identified limitations
print("\n=== IDENTIFIED LIMITATIONS ===")

display(
    workflow_eval.loc[
        workflow_eval["review_status"].eq(
            "Task passed with limitation"
        ),
        [
            "scenario_id",
            "ai_answer",
            "review_limitation"
        ]
    ]
)

print(
    "\nModel risk scores should not be presented "
    "as calibrated probabilities."
)

print("Manual baseline measurements remain pending.")
print("\nWorkflow correctness review completed.")

In [ ]:
# THOMAS: Final workflow evaluation summary

import json
import pandas as pd
import numpy as np

# Verify upstream evaluation results
if "workflow_eval" not in globals():
    raise RuntimeError(
        "Workflow evaluation is missing. "
        "Cells 68 and 69 must be completed first."
    )

# Convert NumPy and Pandas values for JSON serialization
def json_converter(value):

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value) if np.isfinite(value) else None

    if isinstance(value, np.ndarray):
        return value.tolist()

    if value is pd.NA:
        return None

    raise TypeError(
        f"Unsupported JSON type: {type(value).__name__}"
    )


def summarize_workflow_usability(df):

    total = len(df)

    reviewed = df.dropna(
        subset=["ai_correct"]
    )

    successful = df.loc[
        df["execution_success"].eq(True)
    ]

    correct = reviewed.loc[
        reviewed["ai_correct"].eq(True)
    ]

    strict_reviewed = df.dropna(
        subset=["strict_correctness"]
    )

    strict_correct = strict_reviewed.loc[
        strict_reviewed["strict_correctness"].eq(True)
    ]

    complete = bool(
        total > 0 and len(reviewed) == total
    )

    strict_complete = bool(
        total > 0 and len(strict_reviewed) == total
    )

    accuracy = (
        float(len(correct) / total)
        if complete else None
    )

    strict_accuracy = (
        float(len(strict_correct) / total)
        if strict_complete else None
    )

    execution_rate = (
        float(len(successful) / total)
        if total > 0 else None
    )

    ai_median = (
        float(successful["ai_seconds"].median())
        if not successful.empty else None
    )

    ai_mean = (
        float(successful["ai_seconds"].mean())
        if not successful.empty else None
    )

    ai_min = (
        float(successful["ai_seconds"].min())
        if not successful.empty else None
    )

    ai_max = (
        float(successful["ai_seconds"].max())
        if not successful.empty else None
    )

    # Compare tasks completed correctly by both methods
    paired = df.dropna(subset=[
        "baseline_seconds",
        "ai_seconds",
        "baseline_correct",
        "ai_correct"
    ])

    paired_correct = paired.loc[
        paired["baseline_correct"].eq(True)
        & paired["ai_correct"].eq(True)
        & paired["execution_success"].eq(True)
        & paired["baseline_seconds"].gt(0)
    ]

    baseline_median = (
        float(
            paired_correct["baseline_seconds"].median()
        )
        if not paired_correct.empty else None
    )

    paired_ai_median = (
        float(
            paired_correct["ai_seconds"].median()
        )
        if not paired_correct.empty else None
    )

    time_improvement = (
        float(
            (
                (baseline_median - paired_ai_median)
                / baseline_median
            ) * 100
        )
        if baseline_median is not None
        and paired_ai_median is not None
        and baseline_median > 0
        else None
    )

    limitation_count = int(
        df["review_status"]
        .eq("Task passed with limitation")
        .sum()
    )

    manual_baseline_complete = bool(
        total > 0
        and df["baseline_seconds"].notna().all()
        and df["baseline_correct"].notna().all()
        and df["baseline_seconds"].gt(0).all()
    )

    return {
        "evaluation_mode": "Synthetic demo",
        "n_scenarios": int(total),
        "n_reviewed": int(len(reviewed)),
        "n_pending": int(total - len(reviewed)),
        "n_correct": int(len(correct)),
        "n_strictly_correct": int(len(strict_correct)),
        "n_with_limitations": limitation_count,
        "n_successfully_executed": int(len(successful)),
        "execution_success_rate": execution_rate,
        "ai_task_accuracy": accuracy,
        "strict_accuracy": strict_accuracy,
        "ai_median_seconds": ai_median,
        "ai_mean_seconds": ai_mean,
        "ai_min_seconds": ai_min,
        "ai_max_seconds": ai_max,
        "n_paired_correct": int(len(paired_correct)),
        "baseline_median_seconds": baseline_median,
        "paired_ai_median_seconds": paired_ai_median,
        "time_improvement_pct": time_improvement,
        "median_time_improved": (
            bool(paired_ai_median < baseline_median)
            if paired_ai_median is not None
            and baseline_median is not None
            else None
        ),
        "meets_90pct_accuracy_target": (
            bool(accuracy >= 0.90)
            if accuracy is not None else None
        ),
        "meets_90pct_strict_accuracy_target": (
            bool(strict_accuracy >= 0.90)
            if strict_accuracy is not None else None
        ),
        "correctness_review_complete": complete,
        "manual_baseline_complete": manual_baseline_complete
    }


# Generate final evaluation summary
summary = summarize_workflow_usability(
    workflow_eval
)

# Convert all summary values into JSON-compatible types
summary = json.loads(
    json.dumps(
        summary,
        default=json_converter,
        allow_nan=False
    )
)

print("=" * 60)
print("FINAL WORKFLOW EVALUATION")
print("=" * 60)

print(json.dumps(summary, indent=2))

# Performance by specialist workflow
workflow_performance = (
    workflow_eval.groupby("workflow")
    .agg(
        scenarios=("scenario_id", "count"),
        task_accuracy=("ai_correct", "mean"),
        strict_accuracy=("strict_correctness", "mean"),
        execution_success=("execution_success", "mean"),
        median_seconds=("ai_seconds", "median"),
        mean_seconds=("ai_seconds", "mean")
    )
    .reset_index()
)

print("\n=== SPECIALIST WORKFLOW PERFORMANCE ===")

display(workflow_performance)

# Document identified limitations
print("\n=== IDENTIFIED LIMITATIONS ===")

display(
    workflow_eval.loc[
        workflow_eval["review_status"].eq(
            "Task passed with limitation"
        ),
        [
            "scenario_id",
            "review_limitation",
            "ai_seconds"
        ]
    ]
)

# Report manual baseline status
if not summary["manual_baseline_complete"]:

    print(
        "\nManual baseline measurements are pending."
    )

    print(
        "AI-versus-manual time improvement "
        "cannot yet be established."
    )

# Human-readable final results
print("\n=== FINAL RESULTS ===")

print("Total scenarios:", summary["n_scenarios"])
print("Correct task results:", summary["n_correct"])

print(
    "Strictly correct results:",
    summary["n_strictly_correct"]
)

if summary["ai_task_accuracy"] is not None:

    print(
        "Task accuracy:",
        f"{summary['ai_task_accuracy']:.1%}"
    )

if summary["strict_accuracy"] is not None:

    print(
        "Strict correctness:",
        f"{summary['strict_accuracy']:.1%}"
    )

print(
    "Median AI execution time:",
    summary["ai_median_seconds"],
    "seconds"
)

print(
    "Meets 90% task accuracy target:",
    summary["meets_90pct_accuracy_target"]
)

print(
    "Meets 90% strict accuracy target:",
    summary["meets_90pct_strict_accuracy_target"]
)

print("\nWorkflow evaluation summary completed.")